In [ ]:
# algoritmo di enumerazione
import numpy as np

def enumerate_r(A):
    n, m = A.shape
    risultati = []
    

    for i in range(n):  # riga iniziale
        for j in range(m):  # colonna iniziale
            valide = np.ones(m, dtype=bool) # inizializzo vettore valide: a priori tutte le colonne valide
            
            # (i,j) cella iniziale
            for i2 in range(i, n): # espando verso il basso fino a riga i2
                valide &= (A[i2] != 0) # se ci sono zeri nella i2-esima riga, mette False nelle rispettive colonne di valide
                
                for j2 in range(j, m): # espando verso destra fino a colonna j2
                    if valide[j2]:
                        risultati.append((i, j, i2, j2)) # 'vertici sx alto - dx basso' del rettangolo                       
                    else:
                        break 
    return risultati, len(risultati)


# Esempio: istanza proposta dai proff

A = np.ones((4,6)) # definisco la matrice di uni
zero_pos = np.array([[1,1],
                    [1,5],
                    [3,3]])
A[zero_pos[:,0], zero_pos[:,1]] = 0 # metto zeri in posizione porte/finestre

rect, num = enumerate_r(A)

# visualizzazione verticale
for r in rect:
   print(r)

print("Totale rettangoli:", num)

In [ ]:
# creo la matrice per il modello
# le colonne sono i macrorettangoli, le righe le celle
# mij = 1 se macrorettangolo j può coprire cella i, 0 altrimenti

def matrice_binary(A, rect):
    # celle da coprire (solo quelle con valore 1)
    cells=np.argwhere(A==1)
    #cells = [(i, j) for i in range(A.shape[0])
                   # for j in range(A.shape[1])
                   # if A[i, j] == 1]

    n_cells = cells.shape[0]
    #n_cells = len(cells)
    n_rects = len(rect)

    M = np.zeros((n_cells, n_rects), dtype=int)

# se gli indici i,j della cella r sono entrambi compresi rispettivamente 
# tra gli indici i1 12 e j1 j2 del macrorettangolo k, M[r,k]=1
    for k, (i1, j1, i2, j2) in enumerate(rect):
        for r, (i, j) in enumerate(cells):
            if i1 <= i <= i2 and j1 <= j <= j2:
                M[r, k] = 1

    return M, cells

In [ ]:
M, cells = matrice_binary(A,rect)
#print("Celle:", cells)
print("Dimensioni M:", M.shape)
# controllo una colonna della matrice M. es colonna j=21 (ricorda j parte da 0)
print(rect[0:3]) # vedo a che macrorettangolo corrisponde
print(M[:,0:3])

In [ ]:
import gurobipy as gp
from gurobipy import GRB

# creo modello
m = gp.Model()

n_cells, n_rects = M.shape

# variabili
x = m.addVars(n_rects, vtype=GRB.BINARY)



In [ ]:
# fun obiettivo
m.setObjective(gp.quicksum(x[j] for j in range(n_rects)), GRB.MINIMIZE)

In [ ]:
# vincolo
m.addConstrs(gp.quicksum(M[i,j]*x[j] for j in range(n_rects)) == 1 for i in range(n_cells))

In [ ]:
# risolvo
m.optimize()

In [ ]:
# soluzioni
macrorettangoli = [j for j in range(n_rects) if x[j].x > 0.5]

print("Rettangoli selezionati:", macrorettangoli)

# quali e quante (per verifica) celle copre ciascun rettangolo
n_celle_coperte = 0
for j in macrorettangoli:
    print("Rettangolo:", rect[j])
    celle_coperte = np.array([cells[i] for i in range(n_cells) if M[i, j] == 1])
    n_celle_coperte += len(celle_coperte)
    print("Celle coperte:",celle_coperte )

    print()
print ("totale celle coperte:", n_celle_coperte)
